# Biomarker S2 — Inference model d20 cho cả cohort + gom mask thống nhất

Input: `cohort_manifest.csv` từ **S1**. Output: 1 thư mục mask thống nhất
(`/content/drive/MyDrive/knee_biomarkers/masks/{case_id}.nii.gz`), nhãn 8-class union
(1 fem_bone, 2 fem_cart, 3 tib_bone, 4 med_tib_cart, 5 lat_tib_cart, 6 med_meniscus,
7 lat_meniscus, 8 patellar_cart) — dùng thẳng cho **S3** (trích biomarker).

**Ba nguồn mask cho mỗi case (khác với bản gốc — bản này thêm loại 2):**
1. `mask_status == "ready"` → **copy thẳng** label đã có trong `Dataset020_KneeUnion/labelsTr`
   (không tốn compute, không cần GPU). Giống bản gốc.
2. `mask_status == "need_inference"` **và** ảnh gốc lấy từ `Dataset001_KneeOA` (OAIZIB-CM)
   → **[MỚI]** case này thật ra **đã có GT thật cho 5 lớp** (femur, femoral cart., tibia,
   med/lat tibial cart. — đúng khớp thứ tự nhãn 1-5 của Dataset020) trong
   `Dataset001_KneeOA/labelsTr` hoặc `labelsTs`. Ta vẫn chạy nnU-Net (vì model chỉ xuất được
   full 8 lớp, không tách riêng 2 lớp thiếu), nhưng sau đó **ghép**: giữ GT thật cho nhãn 1-5,
   chỉ lấy kết quả AI cho nhãn 6/7/8 (meniscus, patellar cart.) tại các voxel nền (GT=0) —
   không để AI ghi đè lên vùng đã có GT thật.
3. `mask_status == "need_inference"` và **không có GT nào khớp** (vd case gốc từ iMorphics
   chưa lọt vào `Dataset020`) → chạy nnU-Net đầy đủ 8 lớp như bản gốc, không ghép gì thêm.

Chạy theo **chunk** (mặc định 100 ca/lần) để sống sót qua Colab timeout — chạy lại cell inference
nhiều lần, nó tự bỏ qua case đã xong (`skip-existing`).


In [41]:
!pip install -q nnunetv2 SimpleITK

In [42]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 0) Env + config checkpoint (giống hệt merge_s5/merge_s6 - d20 fold 0, 250ep)

In [43]:
import os
from pathlib import Path
import pandas as pd, shutil, re

os.environ["nnUNet_raw"]          = "/content/drive/MyDrive/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = "/content/drive/MyDrive/nnUNet_results"
os.makedirs(os.environ["nnUNet_preprocessed"], exist_ok=True)

RAW  = Path(os.environ["nnUNet_raw"])
D001 = RAW / "Dataset001_KneeOA"     # OAIZIB-CM - co GT that cho 5/8 lop
D020 = RAW / "Dataset020_KneeUnion"  # GT+pseudo 8-lop da gom san (541 ca, dung cho "ready")

TR, PLANS, CHK = "nnUNetTrainer_250epochs", "nnUNetResEncUNetLPlans", "checkpoint_final.pth"
CKPT = Path(os.environ["nnUNet_results"]) / "Dataset020_KneeUnion" / f"{TR}__{PLANS}__3d_fullres" / "fold_0" / CHK
print("checkpoint:", "OK" if CKPT.exists() else "MISSING", CKPT)

BIOM_DIR   = Path("/content/drive/MyDrive/knee_biomarkers")
MASK_DIR   = BIOM_DIR / "masks"          # output thong nhat, dung cho S3
INFER_IN   = Path("/content/infer_in")   # local, input tam cho nnUNetv2_predict
INFER_OUT  = Path("/content/infer_out")  # local, output tam
for d in (MASK_DIR, INFER_IN, INFER_OUT):
    d.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 100   # so ca moi lan chay cell inference (chinh theo quy gio Colab)

# Nhan GT that cua OAIZIB-CM khop dung thu tu 1-5 cua Dataset020 (khong can remap):
# 1 femur, 2 femoral_cart, 3 tibia, 4 med_tib_cart, 5 lat_tib_cart
OAIZIB_GT_CLASSES = [1, 2, 3, 4, 5]
AI_ONLY_CLASSES   = [6, 7, 8]   # med_meniscus, lat_meniscus, patellar_cart - OAIZIB-CM khong co

checkpoint: OK /content/drive/MyDrive/nnUNet_results/Dataset020_KneeUnion/nnUNetTrainer_250epochs__nnUNetResEncUNetLPlans__3d_fullres/fold_0/checkpoint_final.pth


## 1) Load cohort_manifest.csv (từ S1)

In [44]:
cohort = pd.read_csv(BIOM_DIR / "cohort_manifest.csv")
print("tong ca:", len(cohort))
print(cohort["mask_status"].value_counts())

tong ca: 559
mask_status
need_inference    492
ready              67
Name: count, dtype: int64


## 1b) [MỚI] Phân loại `need_inference`: case nào có GT thật (5 lớp) từ OAIZIB-CM

Nhận diện qua chính `dess_path`: nếu ảnh nằm trong `Dataset001_KneeOA/imagesTr` hoặc
`imagesTs` (đặt tên `oaizib_{CMT-ID}_0000.nii.gz`), file GT tương ứng là
`Dataset001_KneeOA/labels{Tr|Ts}/oaizib_{CMT-ID}.nii.gz`. Không cần đọc lại `subInfo` -
tự suy trực tiếp từ đường dẫn ảnh, tránh phụ thuộc thêm file ngoài.

In [45]:
_OAIZIB_IMG_RE = re.compile(r"Dataset001_KneeOA[/\\](images(Tr|Ts))[/\\]oaizib_(\d+)_0000\.nii\.gz$")

def partial_gt_path_for(dess_path: str):
    """Tra ve Path den file GT that (5-lop) neu day la anh OAIZIB-CM, nguoc lai None."""
    m = _OAIZIB_IMG_RE.search(dess_path)
    if not m:
        return None
    split = m.group(2)          # "Tr" hoac "Ts"
    cmt_id = m.group(3)         # vd "007"
    gt_path = D001 / f"labels{split}" / f"oaizib_{cmt_id}.nii.gz"
    return gt_path if gt_path.exists() else None

cohort["partial_gt_path"] = cohort["dess_path"].apply(
    lambda p: str(partial_gt_path_for(p)) if partial_gt_path_for(p) is not None else None
)

need_mask = cohort["mask_status"] == "need_inference"
n_with_gt = cohort.loc[need_mask, "partial_gt_path"].notna().sum()
n_without = need_mask.sum() - n_with_gt
print(f"trong so {need_mask.sum()} ca need_inference:")
print(f"  - co GT that (5 lop, se ghep voi AI cho meniscus/patella): {n_with_gt}")
print(f"  - khong co GT nao (AI du doan du 8 lop nhu ban goc)      : {n_without}")

trong so 492 ca need_inference:
  - co GT that (5 lop, se ghep voi AI cho meniscus/patella): 471
  - khong co GT nao (AI du doan du 8 lop nhu ban goc)      : 21


## 2) Ca `ready` - copy mask từ Dataset020 (không cần GPU), hậu xử lý luôn cho đồng nhất

Nhãn trong `Dataset020` là GT+pseudo (không phải model output trực tiếp) nên hiếm khi có đảo
nhiễu, nhưng vẫn áp `keep_largest_cc` để toàn bộ `MASK_DIR` nhất quán 1 chuẩn xử lý (tránh
biomarker ở S3 bị lệch do 2 nguồn mask xử lý khác nhau).

In [46]:
import SimpleITK as sitk, numpy as np
from scipy.ndimage import label as cc_label

ALL_CLASSES = list(range(1, 9))

def keep_largest_cc(seg, classes):
    out = np.zeros_like(seg)
    for c in classes:
        m = seg == c
        if m.sum() == 0:
            continue
        lab, n = cc_label(m)
        if n <= 1:
            out[m] = c; continue
        sizes = np.bincount(lab.ravel()); sizes[0] = 0
        out[lab == sizes.argmax()] = c
    return out

ready = cohort[cohort["mask_status"] == "ready"]
copied, missing = 0, []
for cid in ready["case_id"]:
    src = D020 / "labelsTr" / f"{cid}.nii.gz"
    dst = MASK_DIR / f"{cid}.nii.gz"
    if dst.exists():
        continue
    if src.exists():
        itk = sitk.ReadImage(str(src))
        arr = sitk.GetArrayFromImage(itk).astype(np.uint8)
        cleaned = keep_largest_cc(arr, ALL_CLASSES)
        out_itk = sitk.GetImageFromArray(cleaned)
        out_itk.CopyInformation(itk)
        sitk.WriteImage(out_itk, str(dst))
        copied += 1
    else:
        missing.append(cid)
print(f"da copy + hau xu ly: {copied} | thieu file goc: {len(missing)}")
if missing[:5]:
    print("vi du thieu:", missing[:5])

da copy + hau xu ly: 0 | thieu file goc: 0


## 3) Ca `need_inference` - build input folder cho nnU-Net (chạy theo chunk)

Cả loại A (có GT thật 5 lớp) và loại B (không có GT) đều phải qua nnU-Net - vì model chỉ xuất
được full 8 lớp, không tách riêng 2 lớp meniscus/patella. Việc ghép GT thật vào diễn ra ở
**bước 5**, sau khi có kết quả AI.

In [47]:
need = cohort[cohort["mask_status"] == "need_inference"].copy()
need = need[~need["case_id"].apply(lambda c: (MASK_DIR / f"{c}.nii.gz").exists())]
print("con lai chua co mask:", len(need))

batch = need.iloc[:CHUNK_SIZE]
for _, row in batch.iterrows():
    dst = INFER_IN / f"{row['case_id']}_0000.nii.gz"
    if not dst.exists():
        shutil.copy(row["dess_path"], dst)
print("da chuan bi input cho", len(batch), "ca ->", INFER_IN)

con lai chua co mask: 0
da chuan bi input cho 0 ca -> /content/infer_in


## 4) Chạy nnUNetv2_predict (checkpoint d20) - chạy lại nếu Colab đứt, tự resume theo case

In [48]:
!nnUNetv2_predict -i /content/infer_in -o /content/infer_out \
    -d 20 -c 3d_fullres -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_250epochs \
    -f 0 -chk checkpoint_final.pth --disable_tta


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 0 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 0 cases that I would like to predict


## 5) Hậu xử lý - [MỚI] ghép GT thật (nếu có) + keep-largest-cc, rồi gom vào `MASK_DIR`

`merge_s8_postprocess.ipynb` đã xác nhận `keep_largest_cc` "miễn phí": Dice không đổi, HD95
giảm rõ (đặc biệt sụn chêm). Áp dụng **sau khi ghép GT**, trên mask cuối cùng - không áp riêng
lẻ trên từng nguồn, để tránh 1 cấu trúc bị cắt rời do ghép 2 nguồn khác gốc.

In [49]:
# map case_id -> partial_gt_path (None neu khong co), tra cuu nhanh trong vong lap
gt_lookup = dict(zip(cohort["case_id"], cohort["partial_gt_path"]))

def merge_gt_and_ai(gt_arr, ai_arr):
    """Giu GT that cho nhan 1-5; chi lay AI cho nhan 6/7/8 (meniscus/patella) tai vung
    con la nen (GT == 0) - khong de AI ghi de len vung da co GT that."""
    out = gt_arr.copy()
    for c in AI_ONLY_CLASSES:
        fill = (ai_arr == c) & (gt_arr == 0)
        out[fill] = c
    return out

moved, merged_with_gt, pure_ai = 0, 0, 0
for p in INFER_OUT.glob("*.nii.gz"):
    cid = p.stem.replace(".nii", "")
    dst = MASK_DIR / f"{cid}.nii.gz"
    if dst.exists():
        continue

    ai_itk = sitk.ReadImage(str(p))
    ai_arr = sitk.GetArrayFromImage(ai_itk).astype(np.uint8)

    gt_path = gt_lookup.get(cid)
    if gt_path:
        gt_itk = sitk.ReadImage(str(gt_path))
        gt_arr = sitk.GetArrayFromImage(gt_itk).astype(np.uint8)
        if gt_arr.shape != ai_arr.shape:
            print(f"CANH BAO: shape lech giua GT va AI cho {cid} "
                  f"(GT={gt_arr.shape}, AI={ai_arr.shape}) -> bo qua ghep, dung AI thuan tuy")
            final_arr = ai_arr
            ref_itk = ai_itk
        else:
            final_arr = merge_gt_and_ai(gt_arr, ai_arr)
            ref_itk = gt_itk   # dung header cua GT (nguon goc, dang tin hon) lam chuan spacing/affine
            merged_with_gt += 1
    else:
        final_arr = ai_arr
        ref_itk = ai_itk
        pure_ai += 1

    cleaned = keep_largest_cc(final_arr, ALL_CLASSES)
    out_itk = sitk.GetImageFromArray(cleaned)
    out_itk.CopyInformation(ref_itk)
    sitk.WriteImage(out_itk, str(dst))
    moved += 1

print(f"da hau xu ly + gom vao MASK_DIR: {moved} "
      f"(ghep voi GT that: {merged_with_gt} | AI thuan tuy: {pure_ai})")

# don input tam de chunk sau khong bi lan
for p in INFER_IN.glob("*.nii.gz"):
    p.unlink()
print("da xoa input tam. San sang cho chunk tiep theo (chay lai cell 3+4+5).")

da hau xu ly + gom vao MASK_DIR: 0 (ghep voi GT that: 0 | AI thuan tuy: 0)
da xoa input tam. San sang cho chunk tiep theo (chay lai cell 3+4+5).


## 6) Kiểm tra tiến độ tổng thể

In [50]:
total = len(cohort)
have = sum((MASK_DIR / f"{c}.nii.gz").exists() for c in cohort["case_id"])
print(f"da co mask: {have}/{total} ca ({have/total*100:.1f}%)")
if have < total:
    print("-> chay lai cell 3 (build input) + cell 4 (predict) + cell 5 (gom) cho den khi du.")
else:
    print("-> XONG. Sang S3 de trich bien so hoc tu MASK_DIR.")

da co mask: 559/559 ca (100.0%)
-> XONG. Sang S3 de trich bien so hoc tu MASK_DIR.


## 7) [MỚI] Hoàn thiện `source_dataset` cho toàn cohort + lưu `mask_provenance.csv`

Nối tiếp cột `source_dataset` đã gán sơ bộ ở S1 (`gt_real_imorphics` / `d020_ready_unclear_provenance`
/ `pending_s2`). Ở đây cập nhật chính xác cho các case `need_inference`:
- Có `partial_gt_path` (OAIZIB-CM) → `partial_gt_oaizib_ai_meniscus_patella` (nhãn 1-5 thật,
  6-8 AI) — độ tin cậy **cao cho thể tích xương/sụn chày/sụn đùi**, nhưng **AI thuần cho
  meniscus/patellar cart.**
- Không có GT nào → `ai_full_8class` (toàn bộ 8 lớp đều là AI dự đoán trên ảnh chưa từng có nhãn).

`mask_provenance.csv` là file tra cứu độ tin cậy theo case, dùng ở **S3** (gắn vào biomarker
table) và **S4/S5** (sensitivity analysis: so kết quả classifier trên tập tin cậy cao vs tập
đầy đủ).

In [51]:
def _final_source(row):
    if row["mask_status"] == "ready":
        return row["source_dataset"]  # gia tri da gan o S1
    return "partial_gt_oaizib_ai_meniscus_patella" if pd.notna(row.get("partial_gt_path")) and row.get("partial_gt_path") not in (None, "None") else "ai_full_8class"

cohort["source_dataset"] = cohort.apply(_final_source, axis=1)
print(cohort["source_dataset"].value_counts())

prov_csv = BIOM_DIR / "mask_provenance.csv"
cohort[["case_id", "mask_status", "source_dataset"]].to_csv(prov_csv, index=False)
# ghi luon lai cohort_manifest.csv de S3/S4/S5 doc duoc cot moi ngay tu file goc
cohort.to_csv(BIOM_DIR / "cohort_manifest.csv", index=False)
print("da luu:", prov_csv)


source_dataset
partial_gt_oaizib_ai_meniscus_patella    471
gt_real_imorphics                         67
ai_full_8class                            21
Name: count, dtype: int64
da luu: /content/drive/MyDrive/knee_biomarkers/mask_provenance.csv


## Ghi chú
- `MASK_DIR` = `/content/drive/MyDrive/knee_biomarkers/masks/` là input DUY NHẤT mà **S3** cần -
  không quan tâm case nào tới từ `ready`, `need_inference`-có-GT, hay `need_inference`-AI-thuần.
- **[MỚI]** Case OAIZIB-CM (`need_inference` nhưng có `partial_gt_path`) giờ dùng **GT thật cho
  femur/tibia/2 sụn chày/sụn đùi**, chỉ AI cho meniscus + patellar cart. - biomarker thể tích
  (`vol_femoral_cart_mm3`, `vol_med_tib_cart_mm3`...) ở `S3` cho các case này giờ đáng tin ngang
  các case iMorphics GT thật, thay vì hoàn toàn phụ thuộc AI như bản gốc.
- Model d20 **chỉ đọc** (predict-only) - không đụng checkpoint gốc, không ảnh hưởng kết quả
  `merge_s5/s6/s7/s8/s9` đã có.
- Nếu thấy `n_with_gt` ở bước 1b bằng 0 dù cohort có nhiều ca OAIZIB-CM: kiểm tra lại
  `Dataset001_KneeOA/labelsTs` có tồn tại không (OAIZIB-CM test split cũng có GT thật, không
  chỉ `labelsTr`) - regex ở bước 1b đã tính cả 2 trường hợp, nhưng đường dẫn thật trên máy bạn
  có thể khác quy ước giả định.
- Nếu cohort rất lớn, cân nhắc bật `--npp`/`--nps` (số process) hoặc chia nhỏ `CHUNK_SIZE`
  hơn nữa để tránh Colab hết RAM khi export.


## 8) [MỚI] Đánh giá chất lượng model d20 trên case có GT thật (Dice nhãn 1-5)

**Lưu ý quan trọng:** không có GT cho meniscus/patellar cart. (nhãn 6-8) trên bất kỳ case
OAIZIB-CM nào, nên **không thể đo Dice trực tiếp cho 2 lớp đó**. Nhưng có thể dùng Dice của
nhãn **1-5** (nơi có GT thật) như một **proxy gián tiếp**: nếu model d20 dự đoán nhãn 1-5 gần
khớp GT thật trên các case này (chưa từng dùng để train, vì đây đúng là tập `need_inference`),
đó là bằng chứng gián tiếp cho thấy model tổng thể hoạt động hợp lý trên domain OAIZIB-CM —
tăng phần nào độ tin cậy cho phần dự đoán 6-8 mà ta không đo trực tiếp được. Đây KHÔNG thay thế
được một đánh giá true Dice cho meniscus/patella, chỉ là chỉ báo gián tiếp tốt hơn không có gì.

Case nào Dice thấp (vd < 0.7) nên được xem lại bằng mắt ở **S6** trước khi tin biomarker.

In [52]:
def dice(a, b, lab):
    ma, mb = (a == lab), (b == lab)
    inter = (ma & mb).sum()
    denom = ma.sum() + mb.sum()
    return float(2 * inter / denom) if denom > 0 else np.nan

dice_rows = []
gt_cases = cohort[cohort["source_dataset"] == "partial_gt_oaizib_ai_meniscus_patella"]
for cid in gt_cases["case_id"]:
    gt_path = gt_lookup.get(cid)
    ai_path = INFER_OUT / f"{cid}.nii.gz"   # INFER_OUT khong bi xoa giua cac chunk (chi INFER_IN bi xoa)
    if not gt_path or not ai_path.exists():
        continue
    gt_arr = sitk.GetArrayFromImage(sitk.ReadImage(str(gt_path))).astype(np.uint8)
    ai_arr = sitk.GetArrayFromImage(sitk.ReadImage(str(ai_path))).astype(np.uint8)
    if gt_arr.shape != ai_arr.shape:
        continue
    row = {"case_id": cid}
    for lab in OAIZIB_GT_CLASSES:
        row[f"dice_class{lab}"] = dice(gt_arr, ai_arr, lab)
    dice_rows.append(row)

if dice_rows:
    dice_df = pd.DataFrame(dice_rows)
    dice_cols = [c for c in dice_df.columns if c.startswith("dice_")]
    print(dice_df[dice_cols].describe())
    low_conf = dice_df[(dice_df[dice_cols].mean(axis=1) < 0.7)]
    print(f"\n{len(low_conf)}/{len(dice_df)} ca co Dice trung binh < 0.7 -> nen xem lai o S6:")
    print(low_conf["case_id"].tolist())
    dice_df.to_csv(BIOM_DIR / "d20_dice_eval.csv", index=False)
    print("\nda luu:", BIOM_DIR / "d20_dice_eval.csv")
else:
    print("Khong tinh duoc Dice: chua co case OAIZIB-CM nao (partial_gt_oaizib_ai_meniscus_patella)")
    print("da qua nnUNetv2_predict trong INFER_OUT o phien nay. INFER_OUT KHONG bi xoa giua cac")
    print("chunk (chi INFER_IN bi xoa) - chay them vai chunk nua roi chay lai cell nay.")


       dice_class1  dice_class2  dice_class3  dice_class4  dice_class5
count   471.000000   471.000000   471.000000   471.000000   471.000000
mean      0.985283     0.895180     0.986880     0.854314     0.869978
std       0.002924     0.026252     0.003138     0.049686     0.052731
min       0.970037     0.786420     0.963431     0.479946     0.579329
25%       0.983865     0.879604     0.985529     0.827210     0.840330
50%       0.985921     0.897635     0.987386     0.860750     0.874874
75%       0.987263     0.915237     0.988897     0.888870     0.911451
max       0.990169     0.939203     0.992189     0.946856     0.954245

0/471 ca co Dice trung binh < 0.7 -> nen xem lai o S6:
[]

da luu: /content/drive/MyDrive/knee_biomarkers/d20_dice_eval.csv
